# Evaluación final del modelo

En este notebook voy a realizar la evaluación final del modelo seleccionado utilizando el periodo de prueba que se mantuvo separado durante todo el desarrollo.

Hasta este momento, los datos desde 2025 no se han utilizado para seleccionar variables, ajustar parámetros ni elegir el modelo.

El modelo seleccionado es una regresión logística con las variables utilizadas durante el desarrollo, excluyendo `Posicion_cierre`. La configuración utiliza `StandardScaler`, una regularización de `C = 0.01`, sin ajuste de pesos entre clases y un umbral de clasificación de 0.50.

Para la evaluación final, el modelo se entrenará utilizando conjuntamente los datos de entrenamiento y validación, ya que ambos periodos ya fueron utilizados durante la etapa de desarrollo. Posteriormente se evaluará una única vez sobre el conjunto de prueba.

Después de observar los resultados de prueba no se modificarán las variables, los parámetros ni el modelo.

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# Localizo la carpeta principal del proyecto
ruta_actual = Path.cwd().resolve()

if ruta_actual.name == "notebooks":
    ruta_proyecto = ruta_actual.parent
else:
    ruta_proyecto = ruta_actual


# Cargo el archivo definitivo con las particiones
archivo_datos = (
    ruta_proyecto
    / "data"
    / "processed"
    / "eurusd_alpha_vantage_particiones.csv"
)

datos = pd.read_csv(
    archivo_datos,
    parse_dates=["Date"],
    index_col="Date"
)

datos = datos.sort_index()

print("Dimensiones del dataset:", datos.shape)
print()
print(datos["Particion"].value_counts())

Dimensiones del dataset: (4980, 21)

Particion
entrenamiento    4052
validacion        522
prueba            405
futuro              1
Name: count, dtype: int64


In [2]:
for particion in [
    "entrenamiento",
    "validacion",
    "prueba",
    "futuro"
]:
    datos_particion = datos[
        datos["Particion"] == particion
    ]

    print(particion.upper())
    print("Filas:", len(datos_particion))
    print("Primera fecha:", datos_particion.index.min())
    print("Última fecha:", datos_particion.index.max())
    print()

ENTRENAMIENTO
Filas: 4052
Primera fecha: 2007-06-19 00:00:00
Última fecha: 2022-12-29 00:00:00

VALIDACION
Filas: 522
Primera fecha: 2022-12-30 00:00:00
Última fecha: 2024-12-30 00:00:00

PRUEBA
Filas: 405
Primera fecha: 2024-12-31 00:00:00
Última fecha: 2026-07-20 00:00:00

FUTURO
Filas: 1
Primera fecha: 2026-07-21 00:00:00
Última fecha: 2026-07-21 00:00:00



In [3]:
# Estas son las variables definitivas del modelo
variables_finales = [
    "Retorno_diario",
    "Retorno_lag_1",
    "Retorno_lag_2",
    "Retorno_lag_3",
    "Retorno_lag_5",
    "Rango_diario",
    "Cuerpo_vela",
    "Distancia_MA5",
    "Distancia_MA10",
    "Distancia_MA20",
    "Volatilidad_5",
    "Volatilidad_20",
    "RSI_14",
    "MACD_hist"
]

print("Número de variables:", len(variables_finales))

print("\nVariables utilizadas:")
for variable in variables_finales:
    print("-", variable)

Número de variables: 14

Variables utilizadas:
- Retorno_diario
- Retorno_lag_1
- Retorno_lag_2
- Retorno_lag_3
- Retorno_lag_5
- Rango_diario
- Cuerpo_vela
- Distancia_MA5
- Distancia_MA10
- Distancia_MA20
- Volatilidad_5
- Volatilidad_20
- RSI_14
- MACD_hist


In [4]:
# Uno entrenamiento y validación para entrenar el modelo definitivo
desarrollo = datos[
    datos["Particion"].isin([
        "entrenamiento",
        "validacion"
    ])
].copy()

# Mantengo completamente separado el periodo de prueba
prueba = datos[
    datos["Particion"] == "prueba"
].copy()


X_desarrollo = desarrollo[
    variables_finales
]

y_desarrollo = desarrollo[
    "Objetivo"
].astype(int)


X_prueba = prueba[
    variables_finales
]

y_prueba = prueba[
    "Objetivo"
].astype(int)


print("Desarrollo:")
print("X:", X_desarrollo.shape)
print("y:", y_desarrollo.shape)

print("\nPrueba final:")
print("X:", X_prueba.shape)
print("y:", y_prueba.shape)

Desarrollo:
X: (4574, 14)
y: (4574,)

Prueba final:
X: (405, 14)
y: (405,)


## Entrenamiento del modelo final

La configuración del modelo quedó seleccionada antes de utilizar el conjunto de prueba.

Voy a entrenar la regresión logística utilizando conjuntamente los datos de entrenamiento y validación. De esta forma aprovecho toda la información que ya formó parte del desarrollo, mientras que el periodo de prueba continúa utilizándose únicamente para la evaluación final.

La configuración utilizada es:

- Regresión logística.
- 14 variables predictoras, excluyendo `Posicion_cierre`.
- Estandarización mediante `StandardScaler`.
- `C = 0.01`.
- Sin pesos adicionales entre clases.
- Umbral de clasificación de 0.50.

In [5]:
# Creo el modelo con la configuración que quedó seleccionada
modelo_final = Pipeline([
    (
        "escalador",
        StandardScaler()
    ),
    (
        "modelo",
        LogisticRegression(
            C=0.01,
            class_weight=None,
            max_iter=1000,
            random_state=42
        )
    )
])

# Entreno únicamente con los datos de desarrollo
modelo_final.fit(
    X_desarrollo,
    y_desarrollo
)

print("Modelo final entrenado correctamente.")

Modelo final entrenado correctamente.


## Predicción sobre el conjunto de prueba

En este punto voy a utilizar por primera vez el modelo seleccionado para generar predicciones sobre el periodo de prueba reservado desde 2025.

Los resultados obtenidos a partir de aquí se consideran resultados finales y no se utilizarán para modificar posteriormente la configuración del modelo.

In [6]:
# Predicciones sobre los datos utilizados para desarrollar el modelo
pred_desarrollo = modelo_final.predict(
    X_desarrollo
)

prob_desarrollo = modelo_final.predict_proba(
    X_desarrollo
)[:, 1]


# Predicción definitiva sobre el conjunto de prueba
pred_prueba = modelo_final.predict(
    X_prueba
)

prob_prueba = modelo_final.predict_proba(
    X_prueba
)[:, 1]

print("Predicciones realizadas.")
print("Predicciones de prueba:", len(pred_prueba))

Predicciones realizadas.
Predicciones de prueba: 405


In [7]:
# Utilizo las mismas métricas que durante el desarrollo
def calcular_metricas(
    nombre,
    particion,
    y_real,
    y_predicho,
    probabilidades=None
):
    resultado = {
        "Modelo": nombre,
        "Particion": particion,
        "Accuracy": accuracy_score(
            y_real,
            y_predicho
        ),
        "Balanced_accuracy": balanced_accuracy_score(
            y_real,
            y_predicho
        ),
        "Precision": precision_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "Recall": recall_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "F1": f1_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "ROC_AUC": None
    }

    if probabilidades is not None:
        resultado["ROC_AUC"] = roc_auc_score(
            y_real,
            probabilidades
        )

    return resultado

## Resultados finales

Para interpretar correctamente el resultado del modelo, voy a compararlo con las mismas referencias sencillas utilizadas durante el desarrollo.

La clase mayoritaria predice siempre la clase más frecuente de los datos de desarrollo.

La persistencia supone que si el EUR/USD subió durante la jornada actual también subirá en la siguiente, y de la misma manera para una bajada.

El modelo final debe superar estas referencias para poder considerar que aporta alguna información adicional.

In [8]:
# Modelo que siempre elige la clase mayoritaria del desarrollo
modelo_mayoria = DummyClassifier(
    strategy="most_frequent"
)

modelo_mayoria.fit(
    X_desarrollo,
    y_desarrollo
)

pred_mayoria_prueba = modelo_mayoria.predict(
    X_prueba
)

prob_mayoria_prueba = modelo_mayoria.predict_proba(
    X_prueba
)[:, 1]


# Persistencia: utilizo la dirección de la jornada actual
pred_persistencia_prueba = (
    prueba["Retorno_diario"] > 0
).astype(int)


resultados_finales = [
    calcular_metricas(
        "Clase mayoritaria",
        "Prueba",
        y_prueba,
        pred_mayoria_prueba,
        prob_mayoria_prueba
    ),
    calcular_metricas(
        "Persistencia",
        "Prueba",
        y_prueba,
        pred_persistencia_prueba
    ),
    calcular_metricas(
        "Regresión logística final",
        "Desarrollo",
        y_desarrollo,
        pred_desarrollo,
        prob_desarrollo
    ),
    calcular_metricas(
        "Regresión logística final",
        "Prueba",
        y_prueba,
        pred_prueba,
        prob_prueba
    )
]


tabla_final = pd.DataFrame(
    resultados_finales
)

columnas_metricas = [
    "Accuracy",
    "Balanced_accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC"
]

tabla_final[columnas_metricas] = (
    tabla_final[columnas_metricas]
    .round(4)
)

display(
    tabla_final.set_index(
        ["Modelo", "Particion"]
    )
)

Accuracy  Balanced_accuracy  Precision  \
Modelo                    Particion                                            
Clase mayoritaria         Prueba        0.4815             0.5000     0.4815   
Persistencia              Prueba        0.5111             0.5104     0.4923   
Regresión logística final Desarrollo    0.5238             0.5235     0.5231   
                          Prueba        0.4840             0.4859     0.4688   

                                      Recall      F1  ROC_AUC  
Modelo                    Particion                            
Clase mayoritaria         Prueba      1.0000  0.6500   0.5000  
Persistencia              Prueba      0.4923  0.4923      NaN  
Regresión logística final Desarrollo  0.5955  0.5570   0.5346  
                          Prueba      0.5385  0.5012   0.4913

In [9]:
distribucion_prueba = (
    y_prueba
    .value_counts()
    .sort_index()
)

print("Distribución real del conjunto de prueba:")
print()

print(
    "Bajadas o cierres iguales:",
    distribucion_prueba.get(0, 0)
)

print(
    "Subidas:",
    distribucion_prueba.get(1, 0)
)

print()

print(
    "Porcentaje de subidas:",
    round(
        y_prueba.mean() * 100,
        2
    ),
    "%"
)

Distribución real del conjunto de prueba:

Bajadas o cierres iguales: 210
Subidas: 195

Porcentaje de subidas: 48.15 %


## Matriz de confusión

Además del porcentaje total de aciertos, voy a comprobar cuántas subidas y bajadas fueron clasificadas correctamente.

Esto permite observar si el modelo funciona de forma relativamente equilibrada o si tiende a predecir principalmente una de las dos direcciones.

In [10]:
matriz = confusion_matrix(
    y_prueba,
    pred_prueba
)

tabla_confusion = pd.DataFrame(
    matriz,
    index=[
        "Real: bajada",
        "Real: subida"
    ],
    columns=[
        "Predicción: bajada",
        "Predicción: subida"
    ]
)

display(tabla_confusion)

,Predicción: bajada,Predicción: subida
Real: bajada,91,119
Real: subida,90,105


## Resultados por año

Después de obtener el resultado conjunto de la prueba, voy a revisar 2025 y 2026 por separado.

Este análisis es únicamente descriptivo. Los resultados de cada año no se utilizarán para cambiar el modelo, sino para comprobar si su comportamiento fue estable durante el periodo final.

In [11]:
# Obtengo la fecha de la jornada que cada fila intenta predecir
fecha_objetivo = (
    datos.index
    .to_series()
    .shift(-1)
)

anio_objetivo_prueba = (
    fecha_objetivo
    .loc[prueba.index]
    .dt.year
)


# Conservo las fechas originales junto a las predicciones
pred_prueba_serie = pd.Series(
    pred_prueba,
    index=prueba.index
)

prob_prueba_serie = pd.Series(
    prob_prueba,
    index=prueba.index
)


resultados_anuales = []

for anio in sorted(
    anio_objetivo_prueba
    .dropna()
    .unique()
):
    mascara_anio = (
        anio_objetivo_prueba == anio
    )

    resultado_anio = calcular_metricas(
        "Regresión logística final",
        str(int(anio)),
        y_prueba.loc[
            mascara_anio
        ],
        pred_prueba_serie.loc[
            mascara_anio
        ],
        prob_prueba_serie.loc[
            mascara_anio
        ]
    )

    resultado_anio["Filas"] = (
        mascara_anio.sum()
    )

    resultados_anuales.append(
        resultado_anio
    )


tabla_anual = pd.DataFrame(
    resultados_anuales
)

tabla_anual = tabla_anual[
    [
        "Modelo",
        "Particion",
        "Filas",
        "Accuracy",
        "Balanced_accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ]
]

tabla_anual[columnas_metricas] = (
    tabla_anual[columnas_metricas]
    .round(4)
)

display(
    tabla_anual.set_index(
        ["Modelo", "Particion"]
    )
)

Filas  Accuracy  Balanced_accuracy  \
Modelo                    Particion                                       
Regresión logística final 2025         261    0.5096             0.5099   
                          2026         144    0.4375             0.4383   

                                     Precision  Recall      F1  ROC_AUC  
Modelo                    Particion                                      
Regresión logística final 2025          0.5067  0.5846  0.5429   0.5174  
                          2026          0.3919  0.4462  0.4173   0.4255

## Distribución de las probabilidades

También voy a revisar el nivel de confianza de las probabilidades generadas por el modelo.

Una clasificación puede ser correcta aunque la probabilidad se encuentre muy cerca del 50 %. Por esta razón, además del porcentaje de aciertos, es útil observar si las predicciones se encuentran generalmente cerca del punto de decisión o si existen jornadas en las que el modelo muestra una mayor diferencia.

In [12]:
resumen_probabilidades = pd.Series(
    prob_prueba,
    name="Probabilidad_subida"
).describe(
    percentiles=[
        0.10,
        0.25,
        0.50,
        0.75,
        0.90
    ]
)

display(
    resumen_probabilidades.round(4)
)

count    405.0000
mean       0.5005
std        0.0244
min        0.3699
10%        0.4741
25%        0.4885
50%        0.5020
75%        0.5150
90%        0.5284
max        0.5659
Name: Probabilidad_subida, dtype: float64

## Conclusión de la evaluación final

La regresión logística seleccionada durante el desarrollo obtuvo en el conjunto de prueba un accuracy del 48,40 %, una balanced accuracy del 48,59 % y un ROC-AUC del 49,13 %.

Estos resultados fueron inferiores a los obtenidos durante la etapa de desarrollo, donde el modelo había alcanzado aproximadamente un 52-53 %, y también quedaron por debajo del modelo sencillo de persistencia, que obtuvo un accuracy del 51,11 %.

El conjunto de prueba estaba bastante equilibrado, con 210 bajadas o cierres iguales y 195 subidas, por lo que el resultado no puede explicarse por una diferencia importante entre las dos clases.

Al analizar los resultados por año, en 2025 el modelo obtuvo una balanced accuracy del 50,99 %, prácticamente equivalente al azar. En 2026 el resultado bajó hasta el 43,83 %. Debe tenerse en cuenta que 2026 corresponde únicamente a una parte del año, ya que los datos disponibles llegan hasta julio.

Las probabilidades generadas por el modelo también se concentraron principalmente alrededor del 50 %. La mediana fue de 0,5020 y la mitad central de las predicciones se situó aproximadamente entre 0,4885 y 0,5150. Esto muestra que el propio modelo generalmente encontró poca diferencia entre la probabilidad de subida y de bajada.

En conjunto, la evaluación final indica que la pequeña capacidad predictiva encontrada durante la etapa de desarrollo no se mantuvo en datos posteriores completamente separados.

Por tanto, con las variables técnicas y los modelos utilizados en este proyecto no se encontró una relación suficientemente estable para predecir de forma fiable la dirección diaria del EUR/USD.

Este resultado también confirma la importancia de mantener un periodo final completamente separado. Si únicamente se hubieran considerado los resultados de entrenamiento o validación, se habría podido concluir que existía una pequeña ventaja predictiva. Sin embargo, al evaluar el modelo sobre datos posteriores no utilizados durante el desarrollo, esa ventaja desapareció.

## Resumen de los principales resultados

La siguiente comparación resume los resultados más importantes obtenidos durante el desarrollo y la evaluación final.

In [14]:
resumen_modelos = pd.DataFrame({
    "Etapa": [
        "Yahoo Finance",
        "Alpha - Clase mayoritaria",
        "Alpha - Persistencia",
        "Alpha - Regresión logística",
        "Alpha - Random Forest ajustado",
        "Alpha - XGBoost ajustado",
        "Evaluación final"
    ],
    "Resultado principal": [
        "79,5 % accuracy",
        "50,0 % balanced accuracy",
        "49,6 % balanced accuracy",
        "53,7 % balanced accuracy",
        "47,6 % balanced accuracy",
        "45,3 % balanced accuracy",
        "48,6 % balanced accuracy"
    ],
    "Interpretacion": [
        "Resultado no reproducido con Alpha Vantage",
        "Referencia sin capacidad predictiva",
        "Resultado cercano al azar",
        "Mejor modelo durante el desarrollo",
        "Sobreajuste y baja generalización",
        "Sobreajuste y baja generalización",
        "La señal no se mantuvo en prueba"
    ]
})

display(resumen_modelos)

,Etapa,Resultado principal,Interpretacion
0,Yahoo Finance,"79,5 % accuracy",Resultado no reproducido con Alpha Vantage
1,Alpha - Clase mayoritaria,"50,0 % balanced accuracy",Referencia sin capacidad predictiva
2,Alpha - Persistencia,"49,6 % balanced accuracy",Resultado cercano al azar
3,Alpha - Regresión logística,"53,7 % balanced accuracy",Mejor modelo durante el desarrollo
4,Alpha - Random Forest ajustado,"47,6 % balanced accuracy",Sobreajuste y baja generalización
5,Alpha - XGBoost ajustado,"45,3 % balanced accuracy",Sobreajuste y baja generalización
6,Evaluación final,"48,6 % balanced accuracy",La señal no se mantuvo en prueba
